# Verificación visual de `pandas_luarix`

Un elemento por sección: se calcula, se imprime la tabla cruda y se dibuja sobre las velas.
**La tabla y el gráfico son la misma información** — si un order block aparece en la tabla
anclado al 2026-03-08, la caja del gráfico tiene que empezar en esa vela.

Qué mirar en cada sección está escrito arriba de cada gráfico, en *Cómo verificar*.

Requisitos: `pip install -r requirements-dev.txt` + `pip install -e .` desde la raíz del repo.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pandas_luarix                      # registra .smc / .indicators / .analysis
from plot_smc import *                    # helpers de dibujo (no son parte de la librería)

pd.set_option("display.width", 140)
plt.rcParams["figure.dpi"] = 110

## 0. Datos

`load_ohlc` lee de `test_data/` (19 símbolos × 4-5 timeframes) y deja el DataFrame en el
formato que espera la librería: índice temporal + columnas `open/high/low/close/volume`
en minúsculas.

Cambia `SYMBOL` / `TIMEFRAME` / `TAIL` y re-ejecuta el notebook entero para verificar
sobre otro activo — es la forma más rápida de detectar que un detector solo "funciona"
en BTC.

In [ ]:
SYMBOL, TIMEFRAME, TAIL = "BTCUSDT", "1d", 260

df = load_ohlc(SYMBOL, TIMEFRAME, TAIL)
print(f"{SYMBOL} {TIMEFRAME} | {df.index[0].date()} -> {df.index[-1].date()} | {len(df)} velas")
df.head()

In [ ]:
# La ventana cruda, sin ningún overlay: la referencia contra la que comparar todo lo demás.
fig, ax = chart(df, f"{SYMBOL} {TIMEFRAME} — velas sin overlays", volume=True)
plt.show()

---
## 1. Swings — `df.smc.swings(length)`

Pivotes confirmados. `length` es el lookback: una vela es swing high si su máximo es el
mayor de las `length` velas a cada lado.

> **Ojo con el desfase confirmación/origen.** Internamente el pivote se *detecta* `length`
> velas después de ocurrir, pero el accessor ya devuelve la vela de **origen** (la del
> extremo real). Es el bug nº1 histórico de esta librería — el gráfico lo verifica: los
> marcadores tienen que caer sobre los picos, no `length` velas a la derecha.

In [ ]:
sw = df.smc.swings(length=10)
print(f"{len(sw)} swings:", sw["swing_type"].value_counts().to_dict())
sw[["high", "low", "swing_type"]].head(10)

In [ ]:
# Cómo verificar: cada triángulo rojo debe estar sobre un máximo local y cada verde
# bajo un mínimo local. Ningún marcador debe quedar desplazado a la derecha del pico.
fig, ax = chart(df, f"Swings (length=10) — {SYMBOL}")
mark_points(ax, df, sw[sw.swing_type == "high"], "high", "v", DOWN, "swing high")
mark_points(ax, df, sw[sw.swing_type == "low"],  "low",  "^", UP,   "swing low")
legend(ax); plt.show()

### El pivote es ASIMÉTRICO — no es `ta.pivothigh`

Esto sorprende a todo el que viene de TradingView. `find_pivots` porta la función `leg()`
de LuxAlgo, cuya condición es:

    high[origen] > max(high[origen+1 .. origen+length])

**Solo mira las `length` velas POSTERIORES.** No exige nada de las anteriores. Un swing
high puede tener velas más altas a su izquierda y seguir siendo válido — con la definición
simétrica de `ta.pivothigh` esos mismos puntos no serían pivotes.

Encima, `find_pivots` solo emite un pivote cuando la pierna *cambia* de sentido, así que
los swings **alternan** high/low: nunca salen dos highs seguidos.

Consecuencia práctica: si comparas la salida contra `ta.pivothigh` de Pine vas a ver
diferencias, y **no son bugs**. La celda siguiente comprueba las dos definiciones para que
la diferencia quede a la vista en números.

In [ ]:
L = 10
for lado, col in (("swing_high", "high"), ("swing_low", "low")):
    rows = getattr(df.smc, lado)(length=L, include_mitigated=True)
    fallan_asim = fallan_sim = 0
    for ts, r in rows.iterrows():
        p = df.index.get_indexer([ts])[0]
        der = df[col].iloc[p + 1: p + L + 1]
        izq = df[col].iloc[max(0, p - L): p]
        if col == "high":
            ok_der = len(der) == 0 or r[col] > der.max()
            ok_izq = len(izq) == 0 or r[col] > izq.max()
        else:
            ok_der = len(der) == 0 or r[col] < der.min()
            ok_izq = len(izq) == 0 or r[col] < izq.min()
        fallan_asim += not ok_der
        fallan_sim  += not (ok_der and ok_izq)
    print(f"{lado}: {len(rows)} pivotes | incumplen def. ASIMÉTRICA (la real): {fallan_asim}"
          f" | incumplen def. simétrica (ta.pivothigh): {fallan_sim}")
    assert fallan_asim == 0, "la librería debe cumplir SIEMPRE su propia definición"

print("\nOK — 0 incumplimientos de la definición real.")
print("Los que 'fallan' la simétrica son la diferencia esperada con ta.pivothigh, no bugs.")

In [ ]:
# Que los swings ALTERNAN: nunca dos highs ni dos lows seguidos.
tipos = df.smc.swings(length=10)["swing_type"].tolist()
seguidos = [(a, b) for a, b in zip(tipos, tipos[1:]) if a == b]
print("pares consecutivos del mismo tipo:", len(seguidos))
assert not seguidos
print("OK — la secuencia alterna high/low como impone el algoritmo de piernas")

---
## 2. Equal Highs / Equal Lows — `df.smc.equal_high(...)`, `.equal_low(...)`

**Devuelven `list[DataFrame]`, no un DataFrame**: cada elemento de la lista es un *grupo*
de pivotes que están al mismo nivel dentro de una tolerancia.

Dos métodos vía `method=`:
- `atr_pivot` (default) — tolerancia = `equal_threshold × ATR(atr_period)`
- `pct_range` — tolerancia = `tolerance_pct` relativo al precio

`length=3` es el default correcto aquí (el resto de la librería usa 50); viene de que en
la referencia el lookback de equal levels es un campo independiente y mucho más corto.

In [ ]:
# atr_period=200 es el default y exige 201 filas; con ventanas cortas hay que bajarlo.
eqh, eql = df.smc.equal_high_low(length=3, atr_period=100, include_mitigated=True)
print(f"{len(eqh)} grupos de equal highs, {len(eql)} de equal lows")
for g in eqh:
    print("  EQH:", [str(i.date()) for i in g.index], "->", g["high"].round(1).tolist())
for g in eql:
    print("  EQL:", [str(i.date()) for i in g.index], "->", g["low"].round(1).tolist())

In [ ]:
# Cómo verificar: los círculos de un mismo grupo deben estar TODOS a la misma altura,
# sobre la horizontal punteada. Si uno queda visiblemente por encima/debajo, la
# tolerancia está mal calculada.
fig, ax = chart(df, f"Equal Highs / Lows (length=3, atr_period=100) — {SYMBOL}")
draw_level_groups(ax, df, eqh, "high", WARN,   "equal high")
draw_level_groups(ax, df, eql, "low",  ACCENT, "equal low")
legend(ax); plt.show()

---
## 3. BOS / CHoCH — `df.smc.bos_choch(length)`

Rupturas de estructura. **BOS** = continuación (rompe en el sentido de la tendencia),
**CHoCH** = cambio de carácter (rompe en contra, primer aviso de giro).

`direction`: `1` alcista, `-1` bajista. La fila es la vela donde el cierre confirma la ruptura.

In [ ]:
ev = df.smc.bos_choch(length=10)
print(f"{len(ev)} eventos:", ev["structure_type"].value_counts().to_dict())
ev[["close", "structure_type", "direction"]]

In [ ]:
# Cómo verificar: un CHoCH debe aparecer donde la tendencia GIRA (tras una secuencia
# de máximos crecientes, el primer evento a la baja), y un BOS donde continúa.
# Dos BOS seguidos en direcciones opuestas sin CHoCH en medio sería un fallo.
fig, ax = chart(df, f"BOS / CHoCH (length=10) — {SYMBOL}")
draw_structure(ax, df, ev)
plt.show()

---
## 4. Order Blocks — `df.smc.order_blocks(...)`

La última vela contraria antes del movimiento impulsivo que rompe estructura. La fila es
esa vela de origen; `top`/`bottom` delimitan la caja.

`include_mitigated=True` los devuelve todos con la columna `mitigated` — la librería
**nunca borra** un elemento consumido, solo lo marca. En el gráfico los mitigados salen
atenuados y con borde punteado.

In [ ]:
ob = df.smc.order_blocks(length=10, include_mitigated=True)
print(f"{len(ob)} order blocks | vigentes: {(~ob['mitigated']).sum()} | mitigados: {ob['mitigated'].sum()}")
ob[["top", "bottom", "direction", "mitigated"]].round(1)

In [ ]:
# Cómo verificar (lo que pediste): la caja debe empezar EXACTAMENTE en la vela de la
# tabla, y su top/bottom deben coincidir con el cuerpo/rango de esa vela. La línea
# vertical gruesa del borde izquierdo marca esa vela de origen.
# Verde = bullish (demanda, bajo el precio), rojo = bearish (oferta, sobre el precio).
fig, ax = chart(df, f"Order Blocks (length=10) — {SYMBOL}")
draw_boxes(ax, df, ob, color_by="direction", dim_col="mitigated", label="order block")
legend(ax); plt.show()

In [ ]:
# Zoom a UN order block concreto para verificarlo vela a vela.
# Cambia `i` para recorrerlos.
i = 0
if len(ob):
    ts = ob.index[i]
    p = df.index.get_indexer([ts])[0]
    win = df.iloc[max(0, p - 12): p + 18]
    fig, ax = chart(win, f"Zoom OB #{i} — origen {ts.date()}  (top={ob.iloc[i]['top']:,.0f} bottom={ob.iloc[i]['bottom']:,.0f})", figsize=(13, 6))
    draw_boxes(ax, win, ob.iloc[[i]], color_by="direction", dim_col="mitigated")
    plt.show()
    print(df.loc[[ts], ["open", "high", "low", "close"]].round(1))

---
## 5. Fair Value Gaps — `df.smc.fair_value_gaps(...)`

Desequilibrio de 3 velas: el rango entre la vela 1 y la 3 que la vela 2 recorrió sin
solapamiento. `filled` marca los ya rellenados por precio posterior.

`min_gap_atr_multiple` filtra por tamaño mínimo real contra ATR — a diferencia de la
referencia LuxAlgo, donde ese umbral estaba hardcodeado a 0 y nunca se calculaba.

In [ ]:
fvg = df.smc.fair_value_gaps(include_filled=True)
print(f"{len(fvg)} FVG | sin rellenar: {(~fvg['filled']).sum()} | rellenados: {fvg['filled'].sum()}")
fvg[["top", "bottom", "direction", "filled"]].head(12).round(1)

In [ ]:
# Cómo verificar: cada caja debe caer en un HUECO real — la mecha de la vela anterior
# y la de la posterior no deben invadir la caja. Si una caja tapa velas completas,
# el detector está mal.
fig, ax = chart(df, f"Fair Value Gaps — {SYMBOL}")
draw_boxes(ax, df, fvg, color_by="direction", dim_col="filled", extend=20, label="FVG")
legend(ax); plt.show()

In [ ]:
# Filtrar por tamaño deja solo los significativos y hace el gráfico legible.
grandes = df.smc.fair_value_gaps(min_gap_atr_multiple=0.5, atr_period=14, include_filled=True)
print(f"{len(grandes)} FVG con hueco >= 0.5 x ATR(14)  (de {len(fvg)} totales)")
fig, ax = chart(df, f"FVG filtrados (>= 0.5 x ATR) — {SYMBOL}")
draw_boxes(ax, df, grandes, color_by="direction", dim_col="filled", extend=25, label="FVG grande")
legend(ax); plt.show()

---
## 6. Liquidity Sweeps — `df.smc.liquidity_sweep(...)`

Stop hunt: una mecha perfora un nivel (swing o equal level) pero el cierre vuelve dentro
en las siguientes `max_confirmation_bars` velas.

`levels=` elige contra qué se barre: `swings`, `equal_high_low` o `both`.

In [ ]:
sweeps = df.smc.liquidity_sweep(length=10)
print(f"{len(sweeps)} sweeps:", sweeps["swept_level_type"].value_counts().to_dict())
sweeps[["high", "low", "swept_level", "swept_level_type", "direction"]].round(1).head(12)

In [ ]:
# Cómo verificar: la X está sobre el nivel barrido y la horizontal va hasta la vela
# del barrido. La MECHA de esa vela debe cruzar la línea, pero su CIERRE debe quedar
# del lado de dentro. Si el cuerpo entero quedó fuera, no era un sweep sino una ruptura.
fig, ax = chart(df, f"Liquidity Sweeps (length=10) — {SYMBOL}")
draw_sweeps(ax, df, sweeps, "sweep")
legend(ax); plt.show()

---
## 7. Fractal y Demand Zone — `df.smc.fractal(length)`, `.demand_zone(...)`

`fractal` devuelve cajas ancladas al pivote con `confirm_time` (cuándo se confirmó) además
de la vela de origen — útil justo para auditar el desfase confirmación/origen.

In [ ]:
fr = df.smc.fractal(length=5, include_mitigated=True)
print(f"{len(fr)} fractales | mitigados: {fr['mitigated'].sum()}")
fr[["top", "bottom", "confirm_time", "direction", "mitigated"]].head(8).round(1)

In [ ]:
# Cómo verificar: `confirm_time` debe ser SIEMPRE posterior al índice de la fila,
# exactamente `length` velas después. Si fueran iguales, habría look-ahead bias.
desfase = (df.index.get_indexer(fr["confirm_time"]) - df.index.get_indexer(fr.index))
print("desfase confirmación - origen (en velas):", sorted(set(desfase)))
assert set(desfase) <= {5}, "el desfase debería ser exactamente length=5"
print("OK — sin look-ahead: el fractal se confirma 5 velas después de su origen")

In [ ]:
fig, ax = chart(df, f"Fractales (length=5) — {SYMBOL}")
draw_boxes(ax, df, fr, color_by="direction", dim_col="mitigated", extend=20, label="fractal")
legend(ax); plt.show()

In [ ]:
dz = df.smc.demand_zone(length=5)
print(f"{len(dz)} demand zones")
fig, ax = chart(df, f"Demand Zones (length=5) — {SYMBOL}")
draw_boxes(ax, df, dz, color=UP, dim_col="mitigated", extend=25, label="demand zone")
legend(ax); plt.show()

---
## 8. IPDA Fractal — `df.smc.ipda_fractal(...)`

Patrones de expansión X-1-2-3-4-5 (impulso) y X-A-B-C (corrección). `compliance` es cuánto
cumple el patrón; `invalidated` marca los que el precio anuló después.

Con ventanas cortas suele devolver 0 filas: necesita `length*2+1` velas y que exista una
secuencia completa de pivotes alternados.

In [ ]:
ipda = df.smc.ipda_fractal(length=10, mode="impulse", direction=0)
print(f"{len(ipda)} expansiones IPDA")
if len(ipda):
    display(ipda[["pattern", "direction", "n_points", "top", "bottom", "compliance", "invalidated"]].round(2))
    fig, ax = chart(df, f"IPDA Fractal (impulse, length=10) — {SYMBOL}")
    draw_boxes(ax, df, ipda, color_by="direction", dim_col="invalidated", label="IPDA")
    legend(ax); plt.show()
else:
    print("Sin resultados en esta ventana — prueba length menor, mode='both' o TAIL mayor.")

---
## 9. Volatilidad — `expansion`, `consolidation`, `fib_expansion`

- `expansion(atr_period, multiplier)` — velas cuyo rango supera `multiplier × ATR`
- `consolidation(window, ...)` — ventanas de rango comprimido
- `fib_expansion(...)` — proyección A-B-C -> D por ratio de Fibonacci

In [ ]:
exp = df.smc.expansion(atr_period=14, multiplier=2.0)
con = df.smc.consolidation(window=10, atr_period=14, multiplier=0.6)
print(f"{len(exp)} velas de expansión | {len(con)} velas en consolidación")

fig, ax = chart(df, f"Expansión (rango > 2 x ATR14) y consolidación — {SYMBOL}")
if len(exp):
    mark_points(ax, df, exp, "high", "v", WARN, "expansión")
if len(con):
    mark_points(ax, df, con, "low", "^", NEUTRAL, "consolidación")
legend(ax); plt.show()

In [ ]:
fib = df.smc.fib_expansion(length=10)
print(f"{len(fib)} proyecciones")
if len(fib):
    display(fib[["a_price", "b_price", "c_price", "ratio", "d_price", "direction"]].round(1).head(8))
    fig, ax = chart(df, f"Fib Expansion — proyección A-B-C -> D — {SYMBOL}")
    for _, r in fib.head(6).iterrows():
        xs = [r["a_pos"], r["b_pos"], r["c_pos"]]
        ys = [r["a_price"], r["b_price"], r["c_price"]]
        col = UP if r["direction"] > 0 else DOWN
        ax.plot(xs, ys, color=col, lw=1.3, marker="o", ms=4, alpha=0.9)
        ax.scatter([r["c_pos"]], [r["d_price"]], marker="*", s=160, color=col, zorder=7)
        ax.annotate(f"D {r['d_price']:,.0f}", (r["c_pos"], r["d_price"]), fontsize=7, color=col, ha="left")
    plt.show()

---
## 10. Indicadores clásicos — `df.indicators.*`

`bb50`, `ema`, `rsi`, `atr`, `stochastic_oscillator`. Devuelven el df completo con columnas
añadidas (no un subconjunto de filas como los elementos SMC).

In [ ]:
bb = df.indicators.bb50(length=20, std=2.0)
ema20 = df.indicators.ema(length=20)["ema"]
ema50 = df.indicators.ema(length=50)["ema"]

fig, ax = chart(df, f"Bollinger(20,2) + EMA 20/50 — {SYMBOL}")
x = np.arange(len(df))
ax.plot(x, bb["upper"],  color=ACCENT,  lw=1.1, label="BB upper")
ax.plot(x, bb["middle"], color=NEUTRAL, lw=1.0, ls="--", label="BB middle")
ax.plot(x, bb["lower"],  color=ACCENT,  lw=1.1, label="BB lower")
ax.fill_between(x, bb["lower"], bb["upper"], color=ACCENT, alpha=0.06)
ax.plot(x, ema20, color=WARN, lw=1.2, label="EMA 20")
ax.plot(x, ema50, color=DOWN, lw=1.2, label="EMA 50")
legend(ax); plt.show()

In [ ]:
# Cómo verificar: la banda media TIENE que ser la SMA(20) exacta — esa es la
# comprobación dura.
print("middle == SMA(20):", np.allclose(bb["middle"].dropna(), df["close"].rolling(20).mean().dropna()))

# El % de cierres dentro de las bandas NO es un test: el 95% de la teoría asume
# retornos normales, y una serie de precios real es tendencial y de colas pesadas.
# En cripto lo normal es 75-90%. Solo alarma si sale muy bajo (<60%) o casi 100%.
dentro = ((df["close"] >= bb["lower"]) & (df["close"] <= bb["upper"])).mean()
print(f"cierres dentro de las bandas: {dentro:.1%}  (típico 75-90% en series reales)")

In [ ]:
rsi = df.indicators.rsi(length=14)["rsi"]
st  = df.indicators.stochastic_oscillator(length=14, d_length=3)
atr = df.indicators.atr(length=14)["atr"]

fig, axes = plt.subplots(4, 1, figsize=(16, 11), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1.2, 1.2, 1.2], "hspace": 0.08})
ax = axes[0]
o, h, l, cl = (df[k].to_numpy(float) for k in ("open", "high", "low", "close"))
xs = np.arange(len(df)); up = cl >= o
ax.vlines(xs[up], l[up], h[up], color=UP, lw=0.8)
ax.vlines(xs[~up], l[~up], h[~up], color=DOWN, lw=0.8)
ax.bar(xs[up],  np.abs(cl - o)[up],  bottom=np.minimum(o, cl)[up],  width=0.7, color=UP)
ax.bar(xs[~up], np.abs(cl - o)[~up], bottom=np.minimum(o, cl)[~up], width=0.7, color=DOWN)
ax.set_title(f"Precio + RSI(14) + Estocástico(14,3) + ATR(14) — {SYMBOL}", loc="left", weight="bold")
ax.grid(alpha=0.15)

axes[1].plot(xs, rsi, color=ACCENT, lw=1.1); axes[1].axhline(70, color=DOWN, ls=":", lw=0.8)
axes[1].axhline(30, color=UP, ls=":", lw=0.8); axes[1].set_ylabel("RSI"); axes[1].grid(alpha=0.15)

axes[2].plot(xs, st["slow_k"], color=ACCENT, lw=1.0, label="%K")
axes[2].plot(xs, st["fast_d"], color=WARN,   lw=1.0, label="%D")
axes[2].axhline(80, color=DOWN, ls=":", lw=0.8); axes[2].axhline(20, color=UP, ls=":", lw=0.8)
axes[2].set_ylabel("Stoch"); axes[2].legend(fontsize=7); axes[2].grid(alpha=0.15)

axes[3].plot(xs, atr, color=NEUTRAL, lw=1.1); axes[3].set_ylabel("ATR"); axes[3].grid(alpha=0.15)
date_ticks(axes[3], df)
plt.show()

---
## 11. Divergencias — `df.analysis.divergence(indicator, length)`

Es **agnóstico del indicador**: acepta cualquier `pd.Series` alineada al índice del df
(RSI, %K del estocástico, CVD, momentum...). Compara los pivotes de esa serie contra el
precio en las mismas velas.

- **regular bearish**: precio hace máximo más alto, indicador más bajo -> agotamiento alcista
- **hidden bearish**: precio máximo más bajo, indicador más alto -> continuación bajista
- **regular bullish** / **hidden bullish**: los simétricos

In [ ]:
dv = df.analysis.divergence(rsi, length=5)
print(f"{len(dv)} divergencias:", dv["divergence_type"].value_counts().to_dict() if len(dv) else {})
dv[["divergence_type", "direction", "prior_price", "price", "prior_indicator_value", "indicator_value"]].round(2)

In [ ]:
# Cómo verificar (automático): la definición de cada tipo impone el signo de las dos
# pendientes. Esto comprueba las 4 combinaciones sin depender del ojo.
REGLAS = {
    "regular_bullish": lambda pp, p, ip, i: p < pp and i > ip,
    "hidden_bullish":  lambda pp, p, ip, i: p > pp and i < ip,
    "regular_bearish": lambda pp, p, ip, i: p > pp and i < ip,
    "hidden_bearish":  lambda pp, p, ip, i: p < pp and i > ip,
}
malas = [r["divergence_type"] for _, r in dv.iterrows()
         if not REGLAS[r["divergence_type"]](r["prior_price"], r["price"],
                                             r["prior_indicator_value"], r["indicator_value"])]
print("divergencias incoherentes con su propia definición:", len(malas))
assert not malas, malas
print("OK — las", len(dv), "divergencias respetan el signo de ambas pendientes")

In [ ]:
# Cómo verificar (visual): las dos líneas de un mismo evento deben tener pendientes
# OPUESTAS — precio arriba / indicador abajo, o al revés. Si van paralelas, es un fallo.
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08})
ax.vlines(xs[up], l[up], h[up], color=UP, lw=0.8)
ax.vlines(xs[~up], l[~up], h[~up], color=DOWN, lw=0.8)
ax.bar(xs[up],  np.abs(cl - o)[up],  bottom=np.minimum(o, cl)[up],  width=0.7, color=UP)
ax.bar(xs[~up], np.abs(cl - o)[~up], bottom=np.minimum(o, cl)[~up], width=0.7, color=DOWN)
ax.set_title(f"Divergencias RSI(14) vs precio — {SYMBOL}", loc="left", weight="bold")
ax.grid(alpha=0.15)

for _, r in dv.iterrows():
    x1 = df.index.get_indexer([r.name])[0]
    x0 = df.index.get_indexer([r["prior_time"]])[0]
    col = UP if r["direction"] > 0 else DOWN
    ax.plot([x0, x1], [r["prior_price"], r["price"]], color=col, lw=1.8, marker="o", ms=5)
    ax2.plot([x0, x1], [r["prior_indicator_value"], r["indicator_value"]], color=col, lw=1.8, marker="o", ms=5)
    ax.annotate(str(r["divergence_type"]).replace("_", " "), (x1, r["price"]),
                fontsize=7, color=col, ha="center", weight="bold")

ax2.plot(xs, rsi, color=NEUTRAL, lw=1.0)
ax2.axhline(70, color=DOWN, ls=":", lw=0.8); ax2.axhline(30, color=UP, ls=":", lw=0.8)
ax2.set_ylabel("RSI"); ax2.grid(alpha=0.15); date_ticks(ax2, df)
plt.show()

---
## 12. Todo junto

Un solo gráfico con los elementos principales. Sirve para ver si se contradicen entre sí:
un order block bullish debería estar cerca de un swing low, un sweep suele preceder a un
CHoCH, etc.

In [ ]:
fig, ax = chart(df, f"{SYMBOL} {TIMEFRAME} — SMC completo", figsize=(17, 8))
draw_boxes(ax, df, ob, color_by="direction", dim_col="mitigated", alpha=0.13, label="order block")
draw_level_groups(ax, df, eqh, "high", WARN, "equal high")
draw_level_groups(ax, df, eql, "low", WARN)
draw_sweeps(ax, df, sweeps, "sweep")
draw_structure(ax, df, ev)
mark_points(ax, df, sw[sw.swing_type == "high"], "high", "v", DOWN, "swing high", size=45)
mark_points(ax, df, sw[sw.swing_type == "low"],  "low",  "^", UP,   "swing low",  size=45)
legend(ax); plt.show()

---
## 13. Barrido multi-símbolo

Un detector que solo funciona en BTC no funciona. Esto corre los detectores sobre varios
símbolos y resume cuántos elementos encuentra cada uno — un 0 en toda una columna es señal
de que algo depende de la escala de precio (BTC ~60.000 vs PEPE ~0,00001).

In [ ]:
SIMBOLOS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "DOGEUSDT", "PEPEUSDT", "XRPUSDT"]

filas = []
for s in SIMBOLOS:
    try:
        d = load_ohlc(s, TIMEFRAME, TAIL)
        h, lo = d.smc.equal_high_low(length=3, atr_period=100, include_mitigated=True)
        filas.append({
            "símbolo": s, "velas": len(d), "precio": f"{d['close'].iloc[-1]:,.6g}",
            "swings": len(d.smc.swings(length=10)),
            "bos/choch": len(d.smc.bos_choch(length=10)),
            "order blocks": len(d.smc.order_blocks(length=10, include_mitigated=True)),
            "FVG": len(d.smc.fair_value_gaps(include_filled=True)),
            "sweeps": len(d.smc.liquidity_sweep(length=10)),
            "EQH+EQL": len(h) + len(lo),
            "divergencias": len(d.analysis.divergence(d.indicators.rsi(length=14)["rsi"], length=5)),
        })
    except Exception as e:
        filas.append({"símbolo": s, "velas": f"ERROR: {type(e).__name__}: {e}"})

pd.DataFrame(filas).set_index("símbolo")